<a href="https://colab.research.google.com/github/omega-u20/SmartCare-HospitalManagement/blob/main/Task3_Preprocessing_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 03 – Data Preprocessing and Feature Engineering


Connecting to google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_FOLDER = '/content/drive/MyDrive/CCS3440 - Asg 2'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

pd.set_option('display.max_columns', None)
df = pd.read_csv(f'{DATA_FOLDER}/smartcare_ai_dataset_1000.csv')

print('Original shape:', df.shape)
df.head()

Original shape: (1000, 33)


,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,missed_previous_appointments,appointment_status,admitted,room_type,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level
0,1,P10001,53,Male,A-,General Medicine,Migraine,2025-04-10,10,1,0,Completed,0,NaN,0,1,127,75,117,211,26.1,0,3,2000,0,0,11596,13596,Paid,Insurance,0,0,High
1,2,P10002,26,Male,B-,General Medicine,Diabetes,2025-05-15,2,3,1,Completed,0,NaN,0,0,130,73,136,173,32.8,0,1,2000,0,0,3652,5652,Paid,Insurance,0,0,Medium
2,3,P10003,22,Male,B+,Orthopedics,Back Pain,2025-07-09,22,7,1,No-Show,0,NaN,0,1,141,64,90,176,29.4,1,0,2500,0,1200,2562,6262,Unpaid,Insurance,1,0,Medium
3,4,P10004,44,Female,AB-,Cardiology,Asthma,2025-10-16,16,1,0,Completed,0,NaN,0,0,124,82,126,189,24.9,2,1,2000,0,5000,10262,17262,Paid,Online,0,0,Medium
4,5,P10005,51,Female,O+,Neurology,Hypertension,2025-12-18,12,4,0,Scheduled,0,NaN,0,1,119,81,65,195,27.0,2,0,4000,0,6000,10414,20414,Paid,Cash,0,0,Medium


## 2. Missing Value Handling

From Task 02, we found only one column with missing values: `room_type`, missing in 906 out of 1000 rows (90.6%). We confirmed this is **not random**  it is missing exactly when `admitted = 0`, because a patient who wasn't admitted was never assigned a room.

**Decision:** We do not drop this column (it still carries useful information for admitted patients)and we do not use mean/mode imputation (which would invent a fake room type). Instead we fill missing values with a new category `'Not Admitted'` which is medically accurate.

In [ ]:
print('Missing values before:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df['room_type'] = df['room_type'].fillna('Not Admitted')

print()
print('Missing values after:')
print(df.isnull().sum().sum(), 'total missing values remaining')

Missing values before:
room_type    906
dtype: int64

Missing values after:
0 total missing values remaining


## 3. Duplicate Record Detection

**Decision:** Check for fully duplicated rows and duplicated `patient_id` values. Task 02 already showed there were none, but we re-verify here as part of the formal cleaning pipeline (good practice, and safer if the dataset changes).

In [ ]:
print('Fully duplicate rows:', df.duplicated().sum())
print('Duplicate patient IDs:', df['patient_id'].duplicated().sum())

df = df.drop_duplicates()
print('Shape after duplicate removal:', df.shape)

Fully duplicate rows: 0
Duplicate patient IDs: 0
Shape after duplicate removal: (1000, 33)


## 4. Outlier Identification

**Method:** We use the IQR (Interquartile Range) method: any value below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` is flagged as a statistical outlier.

**Important distinction:** a statistical outlier is not automatically an *error*. We check whether flagged values are still medically/operationally plausible before deciding what to do with them.

In [ ]:
def iqr_outlier_summary(data, columns):
    rows = []
    for col in columns:
        q1, q3 = data[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = ((data[col] < low) | (data[col] > high)).sum()
        rows.append({'column': col, 'min': data[col].min(), 'max': data[col].max(),
                      'lower_bound': round(low, 1), 'upper_bound': round(high, 1),
                      'num_outliers': n_out})
    return pd.DataFrame(rows).sort_values('num_outliers', ascending=False)

clinical_cols = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi']
iqr_outlier_summary(df, clinical_cols)

,column,min,max,lower_bound,upper_bound,num_outliers
3,blood_sugar_mg_dl,65.0,201.0,44.5,184.5,10
5,bmi,14.0,38.8,14.3,36.7,9
4,cholesterol_mg_dl,100.0,330.0,105.0,305.0,6
1,systolic_bp,85.0,178.0,84.0,172.0,3
2,diastolic_bp,50.0,111.0,51.0,107.0,3
0,age,1.0,90.0,-3.0,93.0,0


**Observation:** The clinical vitals (blood pressure, sugar, cholesterol, BMI) have very few outliers (single digits) and their min/max values are all within medically realistic ranges e.g BMI between 14.0 and 38.8 systolic BP between 85 and 178. These are real patient variation not data errors.

**Decision:** Do not remove any rows for these columns. Removing them would throw away genuine clinical signal that may be useful for predicting readmission (e.g very high blood sugar is a real risk factor not noise).

In [ ]:
financial_cols = ['consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr', 'total_bill_lkr']
iqr_outlier_summary(df, financial_cols)

,column,min,max,lower_bound,upper_bound,num_outliers
1,room_charge_lkr,0,150000,0.0,0.0,94
2,lab_charge_lkr,0,27000,-3450.0,12150.0,61
4,total_bill_lkr,2954,179528,-3760.0,35648.0,59
0,consultation_fee_lkr,1500,4000,-250.0,5750.0,0
3,medicine_charge_lkr,506,20371,-5756.8,21105.2,0


**Observation:** Billing columns show more flagged outliers, but this is expected — charges are naturally right-skewed (most visits are cheap, a handful of ICU/long-stay cases are very expensive). We also verified in Task 02 that `total_bill_lkr` always equals the exact sum of the four charge components, confirming the billing data is internally consistent and not corrupted.

**Decision:** Keep these values as-is. High bills are expected to correlate with admission severity and are potentially useful predictors, not noise to be removed.

In [ ]:
history_cols = ['previous_appointments', 'missed_previous_appointments', 'previous_admissions',
                 'length_of_stay_days', 'lab_tests_count', 'treatments_count']
iqr_outlier_summary(df, history_cols)

,column,min,max,lower_bound,upper_bound,num_outliers
2,previous_admissions,0,5,-1.5,2.5,60
3,length_of_stay_days,0,9,-3.0,5.0,48
4,lab_tests_count,0,10,-2.0,6.0,23
1,missed_previous_appointments,0,4,-1.5,2.5,17
5,treatments_count,0,10,-2.0,6.0,17
0,previous_appointments,0,10,-1.0,7.0,8


**Observation:** These are small integer counts (e.g `previous_admissions` ranges 0–5). The IQR method flags many of them simply because the distribution is tight and skewed toward zero not because the values are unrealistic. A patient with 5 previous admissions is unusual but entirely plausible and is exactly the kind of patient we most want the model to learn from for a readmission task.

**Decision:** Keep all values. No capping or removal these are precisely the high-risk cases the model needs to see.

## 5. Data Cleaning Summary

| Issue | Decision | Reason |
|---|---|---|
| `room_type` missing (90.6%) | Filled with `'Not Admitted'` | Missing is logical (no room if not admitted), not random |
| Duplicate rows | Checked, none found | Confirms one row = one unique visit |
| Outliers (clinical, financial, history) | Kept as-is | Values are plausible and often carry real predictive signal |
| `total_bill_lkr` consistency | Verified against sum of charges | Confirms billing data integrity, no correction needed |


## 6. Feature Selection – Dropping Unneeded Columns

**Decision:** Remove columns that cannot or should not be used as model inputs:
- `record_id`, `patient_id` — unique identifiers, carry no predictive information and could cause the model to memorise individual rows.
- `no_show`, `disease_risk_level` — these are the target variables for the *other two* prediction options (A and C). Leaving them in would let the model "cheat" using information that is not realistically available at prediction time, and they are outside the scope of Option B.

In [ ]:
df_model = df.drop(columns=['record_id', 'patient_id', 'no_show', 'disease_risk_level'])
print('Shape after dropping identifiers and unused targets:', df_model.shape)
df_model.columns.tolist()

Shape after dropping identifiers and unused targets: (1000, 29)


['age',
 'gender',
 'blood_group',
 'department',
 'diagnosis',
 'appointment_date',
 'waiting_days',
 'previous_appointments',
 'missed_previous_appointments',
 'appointment_status',
 'admitted',
 'room_type',
 'length_of_stay_days',
 'previous_admissions',
 'systolic_bp',
 'diastolic_bp',
 'blood_sugar_mg_dl',
 'cholesterol_mg_dl',
 'bmi',
 'lab_tests_count',
 'treatments_count',
 'consultation_fee_lkr',
 'room_charge_lkr',
 'lab_charge_lkr',
 'medicine_charge_lkr',
 'total_bill_lkr',
 'payment_status',
 'payment_method',
 'readmitted_30_days']

## 7. Feature Engineering

We create a small number of new features that are more informative for a readmission model than the raw columns alone, and convert the raw date into something usable.

### 7.1 Date feature extraction
`appointment_date` is a raw date string (e.g. `2025-04-10`). A model cannot use text dates directly, and the exact calendar date itself has little meaning for readmission risk. We extract the **month** and **day of week**, which can capture seasonal or weekly patterns, then drop the original date column.

In [ ]:
df_model['appointment_date'] = pd.to_datetime(df_model['appointment_date'])
df_model['appointment_month'] = df_model['appointment_date'].dt.month
df_model['appointment_dayofweek'] = df_model['appointment_date'].dt.dayofweek  # 0=Monday
df_model = df_model.drop(columns=['appointment_date'])

df_model[['appointment_month', 'appointment_dayofweek']].head()

,appointment_month,appointment_dayofweek
0,4,3
1,5,3
2,7,2
3,10,3
4,12,3


### 7.2 Engineered risk features
Based on the domain knowledge from Task 02 (patients with more hospital history and worse vitals are more likely to be readmitted)  we create a few combined features:

- **`missed_appointment_rate`** — proportion of previous appointments that were missed. This captures patient engagement better than a raw count (a patient with 2 missed out of 3 is very different from 2 missed out of 10).
- **`is_hypertensive`** — flag for blood pressure in a commonly used clinical high range (systolic ≥ 140 or diastolic ≥ 90)turning raw numbers into a clinically meaningful signal.
- **`avg_charge_per_treatment`** — total bill divided by number of treatments indicating cost intensity per treatment given.

In [ ]:
# missed_appointment_rate — guard against divide-by-zero for patients with no previous appointments
df_model['missed_appointment_rate'] = np.where(
    df_model['previous_appointments'] > 0,
    df_model['missed_previous_appointments'] / df_model['previous_appointments'],
    0
)

# is_hypertensive flag
df_model['is_hypertensive'] = ((df_model['systolic_bp'] >= 140) | (df_model['diastolic_bp'] >= 90)).astype(int)

# avg_charge_per_treatment — guard against divide-by-zero
df_model['avg_charge_per_treatment'] = np.where(
    df_model['treatments_count'] > 0,
    df_model['total_bill_lkr'] / df_model['treatments_count'],
    df_model['total_bill_lkr']
)

df_model[['missed_appointment_rate', 'is_hypertensive', 'avg_charge_per_treatment']].describe()

,missed_appointment_rate,is_hypertensive,avg_charge_per_treatment
count,1000.000000,1000.000000,1000.000000
mean,0.181133,0.354000,11529.844746
std,0.269831,0.478448,9790.523560
min,0.000000,0.000000,992.625000
25%,0.000000,0.000000,5685.625000
50%,0.000000,0.000000,9392.000000
75%,0.333333,1.000000,14921.000000
max,1.000000,1.000000,179528.000000


## 8. Feature Encoding

The model needs numbers, not text, so all categorical columns must be encoded.

**Decision:**
- **One-Hot Encoding** for columns with no natural order (nominal): `gender`, `blood_group`, `department`, `diagnosis`, `room_type`, `payment_status`, `payment_method`, `appointment_status`. Each category becomes its own 0/1 column, which avoids implying a false ranking between categories.
- We use `drop_first=True` to avoid redundant columns (dummy variable trap).

In [ ]:
categorical_cols = ['gender', 'blood_group', 'department', 'diagnosis', 'room_type',
                     'payment_status', 'payment_method', 'appointment_status']

print('Cardinality of each categorical column:')
for col in categorical_cols:
    print(f'  {col}: {df_model[col].nunique()} categories')

df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
print()
print('Shape after one-hot encoding:', df_model.shape)

Cardinality of each categorical column:
  gender: 2 categories
  blood_group: 8 categories
  department: 7 categories
  diagnosis: 10 categories
  room_type: 4 categories
  payment_status: 3 categories
  payment_method: 4 categories
  appointment_status: 4 categories

Shape after one-hot encoding: (1000, 59)


## 9. Feature Scaling

Numeric columns are on very different scales (e.g. `age` is 0–90, `total_bill_lkr` is in the thousands). Some algorithms (Logistic Regression, KNN, SVM) are sensitive to this and will let large-scale columns dominate unless we standardise.

**Decision:** Use `StandardScaler` (mean = 0, standard deviation = 1) on the numeric columns. Tree-based models (Decision Tree, Random Forest, XGBoost) don't need scaling but it does no harm to them so we scale once here and keep the scaled version for all models to keep the pipeline consistent.

In [ ]:
numeric_cols = ['age', 'waiting_days', 'previous_appointments', 'missed_previous_appointments',
                 'admitted', 'length_of_stay_days', 'previous_admissions', 'systolic_bp', 'diastolic_bp',
                 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'lab_tests_count', 'treatments_count',
                 'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr',
                 'total_bill_lkr', 'appointment_month', 'appointment_dayofweek',
                 'missed_appointment_rate', 'is_hypertensive', 'avg_charge_per_treatment']

scaler = StandardScaler()
df_model[numeric_cols] = scaler.fit_transform(df_model[numeric_cols])

df_model[numeric_cols].describe().round(2).loc[['mean', 'std']]

,age,waiting_days,previous_appointments,missed_previous_appointments,admitted,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,appointment_month,appointment_dayofweek,missed_appointment_rate,is_hypertensive,avg_charge_per_treatment
mean,0.0,0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


The scaler is fit here for demonstration. When we need to build the actual train/test split in Task 05 we will re-fit the scaler on the **training set only** and apply it to the test set to avoid data leakage. This copy is kept for the EDA and feature-selection steps below.

## 10. Feature Selection – Correlation Check

Before finalising features, we check correlation of numeric features with the target to see which ones look most useful, and check for redundant features that are highly correlated with each other.

In [ ]:
correlations = df_model[numeric_cols + ['readmitted_30_days']].corr()['readmitted_30_days'].drop('readmitted_30_days')
correlations.sort_values(ascending=False)

,readmitted_30_days
admitted,0.829240
length_of_stay_days,0.734844
total_bill_lkr,0.453418
medicine_charge_lkr,0.402495
treatments_count,0.386763
lab_tests_count,0.367357
lab_charge_lkr,0.322054
room_charge_lkr,0.270538
avg_charge_per_treatment,0.181025
previous_admissions,0.064067


**Observation:** As expected from our domain reasoning in Task 02, history-related features (`previous_admissions`, `missed_previous_appointments`) and clinical severity features tend to show the strongest relationship with `readmitted_30_days`. None of the individual correlations are extremely high on their own — this suggests readmission is driven by a *combination* of factors rather than any single variable, which supports using models capable of capturing interactions (Random Forest, XGBoost) rather than relying on one or two features.

In [ ]:
# Check for highly correlated (redundant) feature pairs
corr_matrix = df_model[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
redundant_pairs = [(col, row, upper.loc[row, col]) for col in upper.columns for row in upper.index
                    if upper.loc[row, col] > 0.8]
print('Highly correlated pairs (>0.8):')
for pair in redundant_pairs:
    print(pair)

Highly correlated pairs (>0.8):
('length_of_stay_days', 'admitted', np.float64(0.8340994827201268))
('lab_charge_lkr', 'lab_tests_count', np.float64(0.8760614899372432))
('total_bill_lkr', 'room_charge_lkr', np.float64(0.8801621924907445))


**Observation:** The check found three redundant pairs (correlation > 0.8):
- `length_of_stay_days` & `admitted` (0.83) — expected, since a stay of 0 days almost always means the patient wasn't admitted.
- `lab_charge_lkr` & `lab_tests_count` (0.88) — expected, since lab charges are driven directly by how many tests were run.
- `total_bill_lkr` & `room_charge_lkr` (0.88) — expected, since `total_bill_lkr` is partly built from `room_charge_lkr`.

**Decision:** We keep all of these features for now rather than dropping any, because tree-based models (Random Forest, XGBoost) handle correlated features without much issue, and removing them risks losing information. This is flagged here so it can be revisited in Task 05 if a linear model (e.g. Logistic Regression) shows instability — in that case, one column from each redundant pair could be dropped.

## 11. Final Preprocessed Dataset

In [ ]:
print('Final shape:', df_model.shape)
print('Target column present:', 'readmitted_30_days' in df_model.columns)
df_model.head()

Final shape: (1000, 59)
Target column present: True


,age,waiting_days,previous_appointments,missed_previous_appointments,admitted,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,readmitted_30_days,appointment_month,appointment_dayofweek,missed_appointment_rate,is_hypertensive,avg_charge_per_treatment,gender_Male,blood_group_A-,blood_group_AB+,blood_group_AB-,blood_group_B+,blood_group_B-,blood_group_O+,blood_group_O-,department_General Medicine,department_Laboratory Services,department_Neurology,department_Orthopedics,department_Pediatrics,department_Radiology,diagnosis_Back Pain,diagnosis_Chest Pain,diagnosis_Diabetes,diagnosis_Fever,diagnosis_Fracture,diagnosis_Hypertension,diagnosis_Kidney Infection,diagnosis_Migraine,diagnosis_Pneumonia,room_type_ICU,room_type_Not Admitted,room_type_Private Room,payment_status_Partially Paid,payment_status_Unpaid,payment_method_Cash,payment_method_Insurance,payment_method_Online,appointment_status_Completed,appointment_status_No-Show,appointment_status_Scheduled
0,0.462714,-0.909212,-1.113149,-0.747577,-0.70181,-0.585379,0.151340,-0.091930,-0.381326,0.083372,0.168891,0.136659,-1.365172,0.748782,-0.979965,-0.216653,-1.181812,0.783367,-0.326751,0,-0.712750,0.005055,-0.671621,-0.740262,-0.715115,True,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,True,False,False
1,-1.050523,-1.522974,0.068538,0.604280,-0.70181,-0.585379,-0.892382,0.101879,-0.580556,0.814029,-0.868588,1.712861,-1.365172,-0.504410,-0.979965,-0.216653,-1.181812,-0.933341,-0.906461,0,-0.423602,0.005055,0.564340,-0.740262,-0.600661,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False
2,-1.274706,0.011431,2.431911,0.604280,-0.70181,-0.585379,0.151340,0.812512,-1.477092,-0.954930,-0.786682,0.912997,-0.769547,-1.131006,-0.382789,-0.216653,-0.890582,-1.168891,-0.861947,0,0.154694,-0.500415,-0.141924,1.350873,-0.538325,True,False,False,False,True,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,True,False
3,-0.041698,-0.448890,-1.113149,-0.747577,-0.70181,-0.585379,-0.892382,-0.285739,0.315979,0.429472,-0.431755,-0.145646,-0.173922,-0.504410,-0.979965,-0.216653,0.031647,0.495088,-0.059226,0,1.022139,0.005055,-0.671621,-0.740262,0.585773,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,True,False,False
4,0.350623,-0.755771,0.659381,-0.747577,-0.70181,-0.585379,0.151340,-0.608754,0.216364,-1.916321,-0.267943,0.348388,-0.173922,-1.131006,1.408736,-0.216653,0.274339,0.527935,0.170790,0,1.600435,0.005055,-0.671621,-0.740262,0.907878,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,True


In [ ]:
# Save the cleaned, preprocessed dataset for use in Task 04 (EDA) and Task 05 (Modelling)
df_model.to_csv(f'{DATA_FOLDER}/smartcare_preprocessed_readmission.csv', index=False)
print('Saved: smartcare_preprocessed_readmission.csv')

Saved: smartcare_preprocessed_readmission.csv
